In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import os

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
DATASET_PATH = "../dataset/UNSW_NB15_testing-set.parquet"

dataset = pd.read_parquet(DATASET_PATH)

In [3]:
LIVE_FEATURES = [
    "dur",
    "proto",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
    "smean",
    "dmean"
]

X = dataset[LIVE_FEATURES]
y = dataset["label"]

In [4]:
print(X.shape)
print(X.columns)
print(y.value_counts())

(82332, 13)
Index(['dur', 'proto', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload',
       'dload', 'sinpkt', 'dinpkt', 'smean', 'dmean'],
      dtype='str')
label
1    45332
0    37000
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [7]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())
print(y_test.value_counts())

(65865, 13)
(16467, 13)
label
1    36265
0    29600
Name: count, dtype: int64
label
1    9067
0    7400
Name: count, dtype: int64


In [8]:
X_train = pd.get_dummies(
    X_train,
    columns=["proto"],
    drop_first=False
)

X_test = pd.get_dummies(
    X_test,
    columns=["proto"],
    drop_first=False
)

In [9]:
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)
print(X_train.shape)
print(X_test.shape)

print(
    list(X_train.columns) == list(X_test.columns)
)

(65865, 143)
(16467, 143)
True


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(65865, 143)
(16467, 143)


In [11]:
sequence_length = 10
def create_sequences(X, y, sequence_length=10):

    X_seq = []
    y_seq = []

    for i in range(len(X) - sequence_length + 1):

        X_seq.append(
            X[i:i + sequence_length]
        )

        y_seq.append(
            y.iloc[i + sequence_length - 1]
        )

    return np.array(X_seq), np.array(y_seq)

In [12]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test,
    sequence_length
)

In [13]:
print(X_train_seq.shape)
print(y_train_seq.shape)

print(X_test_seq.shape)
print(y_test_seq.shape)

(65856, 10, 143)
(65856,)
(16458, 10, 143)
(16458,)


In [14]:
X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_seq,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_seq,
    dtype=torch.float32
)

In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [16]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 10, 143])
torch.Size([128])


In [17]:
class LSTMModel(nn.Module):

    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [18]:
input_size = X_train_tensor.shape[2]

model = LSTMModel(
    input_size=input_size,
    hidden_size=32,
    num_layers=1
)

print(model)

LSTMModel(
  (lstm): LSTM(143, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [19]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [20]:
epochs = 10

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 0.5219
Epoch [2/10], Loss: 0.4213
Epoch [3/10], Loss: 0.3968
Epoch [4/10], Loss: 0.3852
Epoch [5/10], Loss: 0.3779
Epoch [6/10], Loss: 0.3721
Epoch [7/10], Loss: 0.3673
Epoch [8/10], Loss: 0.3631
Epoch [9/10], Loss: 0.3598
Epoch [10/10], Loss: 0.3556


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch).squeeze(1)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).int()

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

In [22]:
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1 Score:", f1_score(all_labels, all_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Accuracy: 0.7987604812249361
Precision: 0.8424249642686994
Recall: 0.7805120282498345
F1 Score: 0.8102875472562722

Confusion Matrix:
[[6073 1323]
 [1989 7073]]


In [23]:
LIVE_LSTM_DIR = "../deeplearn_models/live"

os.makedirs(LIVE_LSTM_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(LIVE_LSTM_DIR, "live_lstm_model.pth")
)

joblib.dump(
    scaler,
    os.path.join(LIVE_LSTM_DIR, "live_lstm_scaler.pkl")
)

joblib.dump(
    X_train.columns.tolist(),
    os.path.join(LIVE_LSTM_DIR, "live_lstm_features.pkl")
)

['../deeplearn_models/live\\live_lstm_features.pkl']

In [24]:
config = {
    "input_size": X_train_tensor.shape[2],
    "hidden_size": 32,
    "num_layers": 1,
    "sequence_length": 10,
    "threshold": 0.5
}

joblib.dump(
    config,
    os.path.join(LIVE_LSTM_DIR, "live_lstm_config.pkl")
)

print("Live LSTM files saved successfully")

Live LSTM files saved successfully
